# AqSolDB Experiments Overview

This notebook brings the project into one readable place. It walks through:

- dataset loading and cleaning
- tabular preprocessing for classical ML and DNN models
- graph preprocessing for graph neural networks
- regression experiments
- hyperparameter tuning scripts
- binary-classification ablation studies
- how saved outputs are organized and compared

The code mirrors the repository scripts instead of rewriting the pipeline from scratch, so the notebook stays aligned with the actual project implementation.

## 1. Setup

This cell makes the repository importable whether the notebook is launched from the repo root or from the `notebooks/` folder.

In [4]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "utils").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "utils").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Could not locate the project root from the current notebook session.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import (
    DEFAULT_DESCRIPTOR_NAMES,
    build_classical_feature_matrix,
    build_graph_dataset,
    load_dataset,
    make_binary_labels,
    split_classical_data,
)
from utils.metrics import classification_metrics, regression_metrics
from utils.project_paths import DATASET_PATH, OUTPUTS_DIR

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path:  {DATASET_PATH}")
print(f"Outputs path:  {OUTPUTS_DIR}")

Project root: /home/priyank1/Palash/FOML_GOD/FoML_Project
Dataset path:  /home/priyank1/Palash/FOML_GOD/FoML_Project/data/curated-solubility-dataset.csv
Outputs path:  /home/priyank1/Palash/FOML_GOD/FoML_Project/outputs


## 2. Load and clean the dataset

`load_dataset()` is the shared cleaning entrypoint used across the project. It:

- reads the AqSolDB CSV
- checks that `Name`, `SMILES`, and `Solubility` exist
- coerces `Solubility` to numeric
- drops invalid targets
- validates SMILES strings with RDKit
- creates a `CanonicalSMILES` column for the cleaned molecules

In [5]:
raw_frame = pd.read_csv(DATASET_PATH)
clean_frame = load_dataset()

dataset_summary = pd.Series(
    {
        "raw_rows": len(raw_frame),
        "clean_rows": len(clean_frame),
        "dropped_rows": len(raw_frame) - len(clean_frame),
        "solubility_mean": float(clean_frame["Solubility"].mean()),
        "solubility_std": float(clean_frame["Solubility"].std()),
        "solubility_min": float(clean_frame["Solubility"].min()),
        "solubility_max": float(clean_frame["Solubility"].max()),
    }
)

display(dataset_summary.to_frame("value"))
display(clean_frame.head())

[12:00:45] Explicit valence for atom # 5 N, 4, is greater than permitted
[12:00:45] Explicit valence for atom # 5 N, 4, is greater than permitted


,value
raw_rows,9982.000000
clean_rows,9980.000000
dropped_rows,2.000000
solubility_mean,-2.890454
solubility_std,2.368024
solubility_min,-13.171900
solubility_max,2.137682


,ID,Name,InChI,InChIKey,SMILES,Solubility,SD,Ocurrences,Group,MolWt,MolLogP,MolMR,HeavyAtomCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT,CanonicalSMILES
0,A-3,"N,N,N-trimethyloctadecan-1-aminium bromide",InChI=1S/C21H46N.BrH/c1-5-6-7-8-9-10-11-12-13-...,SZEMGTQCPRNXEG-UHFFFAOYSA-M,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127,0.0,1,G1,392.510,3.9581,102.4454,23.0,0.0,0.0,2.0,17.0,142.0,0.0,0.0,0.0,0.0,0.00,158.520601,0.000000e+00,210.377334,CCCCCCCCCCCCCCCCCC[N+](C)(C)C.[Br-]
1,A-4,Benzo[cd]indol-2(1H)-one,InChI=1S/C11H7NO/c13-11-8-5-1-3-7-4-2-6-9(12-1...,GPYLCFQEKPUWLD-UHFFFAOYSA-N,O=C1Nc2cccc3cccc1c23,-3.254767,0.0,1,G1,169.183,2.4055,51.9012,13.0,1.0,1.0,2.0,0.0,62.0,2.0,0.0,1.0,3.0,29.10,75.183563,2.582996e+00,511.229248,O=C1Nc2cccc3cccc1c23
2,A-5,4-chlorobenzaldehyde,InChI=1S/C7H5ClO/c8-7-3-1-6(5-9)2-4-7/h1-5H,AVPYQKSLYISFPO-UHFFFAOYSA-N,Clc1ccc(C=O)cc1,-2.177078,0.0,1,G1,140.569,2.1525,36.8395,9.0,1.0,0.0,2.0,1.0,46.0,1.0,0.0,0.0,1.0,17.07,58.261134,3.009782e+00,202.661065,O=Cc1ccc(Cl)cc1
3,A-8,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",InChI=1S/2C23H22O3.Zn/c2*1-15(17-9-5-3-6-10-17...,XTUPUYCJWKHGSW-UHFFFAOYSA-L,[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409,0.0,1,G1,756.226,8.1161,200.7106,53.0,6.0,2.0,7.0,10.0,264.0,6.0,0.0,0.0,6.0,120.72,323.755434,2.322963e-07,1964.648666,CC(c1ccccc1)c1cc(C(=O)[O-])c(O)c(C(C)c2ccccc2)...
4,A-9,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,InChI=1S/C25H30N2O4/c1-5-20(26(10-22-14-28-22)...,FAUAZXVRLVIARB-UHFFFAOYSA-N,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065,0.0,1,G1,422.525,2.4854,119.0760,31.0,6.0,0.0,6.0,12.0,164.0,2.0,4.0,4.0,6.0,56.60,183.183268,1.084427e+00,769.899934,c1cc(N(CC2CO2)CC2CO2)ccc1Cc1ccc(N(CC2CO2)CC2CO...


## 3. Tabular preprocessing

Classical models and dense neural networks use fixed-length vectors.

The repository supports three feature modes:

- `fingerprint`: Morgan fingerprint only
- `descriptor`: RDKit descriptors only
- `combined`: fingerprint + descriptors

By default the main regression scripts use `combined` features.

In [6]:
tabular_rows = []
tabular_artifacts = {}

for feature_mode in ("fingerprint", "descriptor", "combined"):
    X, y, feature_frame, feature_names = build_classical_feature_matrix(clean_frame, feature_mode=feature_mode)
    tabular_artifacts[feature_mode] = {
        "X": X,
        "y": y,
        "frame": feature_frame,
        "feature_names": feature_names,
    }
    tabular_rows.append(
        {
            "feature_mode": feature_mode,
            "rows": X.shape[0],
            "num_features": X.shape[1],
            "target_dtype": str(y.dtype),
        }
    )

display(pd.DataFrame(tabular_rows))
print("Default descriptors:")
print(list(DEFAULT_DESCRIPTOR_NAMES))
print("\nFirst 15 combined feature names:")
print(tabular_artifacts["combined"]["feature_names"][:15])

,feature_mode,rows,num_features,target_dtype
0,fingerprint,9980,1024,float32
1,descriptor,9980,12,float32
2,combined,9980,1036,float32


Default descriptors:
['NumHDonors', 'TPSA', 'NumRotatableBonds', 'BertzCT', 'RingCount', 'NumAromaticRings', 'NumValenceElectrons', 'LabuteASA', 'HeavyAtomCount', 'MolWt', 'MolMR', 'MolLogP']

First 15 combined feature names:
['fp_0000', 'fp_0001', 'fp_0002', 'fp_0003', 'fp_0004', 'fp_0005', 'fp_0006', 'fp_0007', 'fp_0008', 'fp_0009', 'fp_0010', 'fp_0011', 'fp_0012', 'fp_0013', 'fp_0014']


## 4. Graph preprocessing

Graph models convert each molecule into a PyTorch Geometric `Data` object.

- nodes = atoms
- edges = bonds
- `x` = node features
- `edge_index` = graph connectivity
- `edge_attr` = encoded bond weights
- `y` = molecule solubility

The main regression scripts use the `full` node feature variant.

In [7]:
graph_rows = []
graph_artifacts = {}

for feature_variant in ("atomic_number", "atomic_number_degree", "full"):
    dataset = build_graph_dataset(clean_frame, feature_variant=feature_variant)
    graph_artifacts[feature_variant] = dataset
    node_counts = np.array([graph.x.shape[0] for graph in dataset], dtype=np.int32)
    edge_counts = np.array([graph.edge_index.shape[1] for graph in dataset], dtype=np.int32)
    graph_rows.append(
        {
            "feature_variant": feature_variant,
            "graphs": len(dataset),
            "node_feature_dim": int(dataset[0].x.shape[1]),
            "avg_nodes": float(node_counts.mean()),
            "median_nodes": float(np.median(node_counts)),
            "avg_directed_edges": float(edge_counts.mean()),
            "median_directed_edges": float(np.median(edge_counts)),
        }
    )

display(pd.DataFrame(graph_rows))
sample_graph = graph_artifacts["full"][0]
print(sample_graph)
print("Node feature matrix shape:", tuple(sample_graph.x.shape))
print("Edge index shape:", tuple(sample_graph.edge_index.shape))
print("Edge attr shape:", tuple(sample_graph.edge_attr.shape))

/home/priyank1/Palash/FOML_GOD/FoML_Project/venv/lib/python3.13/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()
/home/priyank1/Palash/FOML_GOD/FoML_Project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,feature_variant,graphs,node_feature_dim,avg_nodes,median_nodes,avg_directed_edges,median_directed_edges
0,atomic_number,9980,1,17.382966,15.0,35.266934,30.0
1,atomic_number_degree,9980,2,17.382966,15.0,35.266934,30.0
2,full,9980,3,17.382966,15.0,35.266934,30.0


Data(x=[23, 3], edge_index=[2, 42], edge_attr=[42, 1], y=[1], name='N,N,N-trimethyloctadecan-1-aminium bromide', smiles='[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C')
Node feature matrix shape: (23, 3)
Edge index shape: (2, 42)
Edge attr shape: (42, 1)


## 5. Shared split logic

All main regression experiments use a train/test split with `test_size=0.2` and `random_state=42`.

- classical models: train directly on the train split
- dense and graph neural models: split train again into fit/validation subsets
- ablation scripts use stratified splits because the target becomes binary

In [8]:
X_combined = tabular_artifacts["combined"]["X"]
y_combined = tabular_artifacts["combined"]["y"]
frame_combined = tabular_artifacts["combined"]["frame"]

X_train, X_test, y_train, y_test, frame_train, frame_test = split_classical_data(
    X_combined,
    y_combined,
    frame_combined,
    test_size=0.2,
    random_state=42,
)

split_summary = pd.Series(
    {
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "feature_dim": X_train.shape[1],
    }
)
display(split_summary.to_frame("value"))

,value
train_rows,7984
test_rows,1996
feature_dim,1036


## 6. Regression experiments

This section uses the same functions as the scripts in `train/`:

- `train_linear_regression.py`
- `train_gaussian_process.py`
- `train_mlp_regressor.py`
- `train_dense_regressor.py`
- `train_graph_cn.py`
- `train_graph_net.py`
- `train_graph_sage.py`
- `train_graph_mp.py`

Set the flags below depending on how much you want to run in the notebook. `GaussianProcessRegressor` and the larger neural experiments can take noticeably longer.

In [9]:
from train.train_dense_regressor import train_and_evaluate as train_dense_regressor
from train.train_gaussian_process import train_and_evaluate as train_gaussian_process
from train.train_graph_cn import train_and_evaluate as train_graph_cn
from train.train_graph_mp import train_and_evaluate as train_graph_mp
from train.train_graph_net import train_and_evaluate as train_graph_net
from train.train_graph_sage import train_and_evaluate as train_graph_sage
from train.train_linear_regression import train_and_evaluate as train_linear_regression
from train.train_mlp_regressor import train_and_evaluate as train_mlp_regressor

RUN_FULL_REGRESSION = False
RUN_GAUSSIAN_PROCESS = False

regression_results = []

if RUN_FULL_REGRESSION:
    regression_results.append({"model_name": "linear_regression", **train_linear_regression()})
    regression_results.append({"model_name": "mlp_regressor", **train_mlp_regressor()})
    regression_results.append({"model_name": "dense_regressor", **train_dense_regressor()})
    regression_results.append({"model_name": "graph_cn", **train_graph_cn()})
    regression_results.append({"model_name": "graph_net", **train_graph_net()})
    regression_results.append({"model_name": "graph_sage", **train_graph_sage()})
    regression_results.append({"model_name": "graph_mp", **train_graph_mp()})

if RUN_GAUSSIAN_PROCESS:
    regression_results.append({"model_name": "gaussian_process", **train_gaussian_process()})

if regression_results:
    display(pd.DataFrame(regression_results).sort_values("rmse"))
else:
    print("Set RUN_FULL_REGRESSION and/or RUN_GAUSSIAN_PROCESS to True to execute the regression experiments.")

Set RUN_FULL_REGRESSION and/or RUN_GAUSSIAN_PROCESS to True to execute the regression experiments.


## 7. Hyperparameter tuning studies

The tuning scripts save CSV summaries under `outputs/tuning/`.

- `tune_classical_models.py` evaluates linear regression as a baseline and runs randomized search for GPR and MLP
- `tune_dnn.py` compares a small list of dense-regressor architectures
- `tune_graph_models.py` compares graph architectures across a small search space

In [13]:
from tuning.tune_classical_models import main as tune_classical_models
from tuning.tune_dnn import main as tune_dnn
from tuning.tune_graph_models import main as tune_graph_models

RUN_TUNING = True

if RUN_TUNING:
    tune_classical_models()
    tune_dnn()
    tune_graph_models()
else:
    print("Set RUN_TUNING = True to run the tuning scripts and save their CSV summaries under outputs/tuning/.")

[12:20:23] Explicit valence for atom # 5 N, 4, is greater than permitted
[12:20:23] Explicit valence for atom # 5 N, 4, is greater than permitted


KeyboardInterrupt: 

## 8. Ablation studies

The ablation folder changes the problem from regression to binary classification using:

`label = 1 if Solubility >= -3.0 else 0`

Two ablation scripts exist:

- `run_feature_ablation.py`: compares tabular feature modes and tabular models
- `run_graph_ablation.py`: compares graph node-feature variants and graph architectures

In [11]:
from ablation.run_feature_ablation import main as run_feature_ablation
from ablation.run_graph_ablation import main as run_graph_ablation

threshold = -3.0
labels = make_binary_labels(clean_frame["Solubility"], threshold=threshold)
print(f"Positive rate at threshold {threshold}: {labels.mean():.4f}")

RUN_ABLATION = False

if RUN_ABLATION:
    run_feature_ablation()
    run_graph_ablation()
else:
    print("Set RUN_ABLATION = True to execute the classification ablation scripts.")

Positive rate at threshold -3.0: 0.5696
Set RUN_ABLATION = True to execute the classification ablation scripts.


## 9. Read saved outputs in one place

After running any training, tuning, or ablation cells above, this section helps collect the saved CSV and JSON files from `outputs/` for comparison.

In [12]:
def collect_metrics_json(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame()

    for metrics_path in root.rglob("metrics.json"):
        payload = json.loads(metrics_path.read_text(encoding="utf-8"))
        rows.append(
            {
                "relative_path": str(metrics_path.relative_to(root)),
                **payload,
            }
        )
    return pd.DataFrame(rows)


metrics_frame = collect_metrics_json(OUTPUTS_DIR)
if metrics_frame.empty:
    print("No saved experiment metrics were found yet. Run some experiment cells first.")
else:
    sort_column = "rmse" if "rmse" in metrics_frame.columns else metrics_frame.columns[0]
    display(metrics_frame.sort_values(sort_column))

No saved experiment metrics were found yet. Run some experiment cells first.


## 10. Suggested notebook workflow

A practical order for using this notebook is:

1. Run the setup and preprocessing sections.
2. Inspect the tabular and graph representations.
3. Run the regression experiments you want to discuss first.
4. Run tuning only after the base models are clear.
5. Run the ablation section when you want to explain the classification study separately.
6. Use the final section to compare saved outputs from all runs.